# Objective:
 Predict Plus-Minus, using points, turnovers, field goals attempted, and assists

 Plus-minus: tracks a team's net point differential—how many more points the team scores than the opponent—while a specific player is on the court. A positive number means the team outscored the opponent, while a negative number means they were outscored

In [5]:
from nba_api.stats.endpoints import leaguedashplayerstats
import pandas as pd

# Pull all players' per-game stats for a recent regular season
data = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2023-24',
    season_type_all_star='Regular Season',
    per_mode_detailed='PerGame',
)

df = data.get_data_frames()[0]
print(df.shape)
df.head()



(572, 67)


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612742,DAL,23.0,42,27,15,0.643,...,404,496,478,442,214,483,257,38,482,1
1,1631260,AJ Green,AJ,1610612749,MIL,24.0,56,35,21,0.625,...,513,444,464,363,178,455,257,38,425,1
2,1631100,AJ Griffin,AJ,1610612737,ATL,20.0,20,8,12,0.400,...,442,533,544,478,439,516,257,38,509,1
3,203932,Aaron Gordon,Aaron,1610612743,DEN,28.0,73,49,24,0.671,...,77,167,60,110,17,93,54,38,105,1
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,27.0,78,39,39,0.500,...,337,250,325,278,164,338,257,38,327,1


In [ ]:
filtered_df = df[(df['GP'] >= 30) & (df['MIN'] >= 15)] 
# filters out players that dont play more than 30 games
# and players that play less than 15 min per game
# cleans things up
X = pd.DataFrame(filtered_df[['PTS', 'TOV', 'FGA', 'AST']])
y = pd.DataFrame(filtered_df.PLUS_MINUS)
#organizing data 

In [17]:

X.describe()
#prints as expected

,PTS,TOV,FGA,AST
count,301.000000,301.000000,301.000000,301.000000
mean,12.655814,1.420598,9.795017,2.943189
std,6.441367,0.771173,4.679388,1.964840
min,1.700000,0.300000,1.600000,0.500000
25%,7.600000,0.800000,6.200000,1.400000
50%,11.000000,1.200000,8.500000,2.400000
75%,16.500000,1.800000,12.900000,4.100000
max,34.700000,4.400000,23.600000,10.900000


In [18]:
y.describe()  

,PLUS_MINUS
count,301.000000
mean,0.122591
std,3.542112
min,-9.200000
25%,-2.100000
50%,0.200000
75%,2.600000
max,8.600000


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"train: {X_train.shape}, test: {X_test.shape}")

train: (240, 4), test: (61, 4)


Allocates data to two groups, one for training (457), and one for testing (115) - uses entire data set


*editted* 


 After filtering, (240) & (61).  This shows how many players dont get many minutes/games 
 important because a player that plays a low amount of minutes may only play "garbage time" 

In [21]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# Three trees with very different complexity
trees = {
    "shallow (max_depth=3)": DecisionTreeRegressor(max_depth=3, random_state=42),
    "medium (max_depth=8)": DecisionTreeRegressor(max_depth=8, random_state=42),
    "giant (no limit)":     DecisionTreeRegressor(random_state=42),
}

print(f"{'tree':<25} | {'leaves':>7} | {'train MSE':>10} | {'test MSE':>10}")
print("-" * 65)
for name, tree in trees.items():
    tree.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, tree.predict(X_train))
    test_mse  = mean_squared_error(y_test,  tree.predict(X_test))
    print(f"{name:<25} | {tree.get_n_leaves():>7} | {train_mse:>10.4f} | {test_mse:>10.4f}")

tree                      |  leaves |  train MSE |   test MSE
-----------------------------------------------------------------
shallow (max_depth=3)     |       8 |     9.9530 |    12.6851
medium (max_depth=8)      |      43 |     5.2397 |    13.1091
giant (no limit)          |     235 |     0.0000 |    17.3038


Looks like medium is the best option out of the three
Note Train: 5.24 and Test: 13.111

next: utilize pruning on giant unpruned tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Train one tree per sampled alpha, record train and test MSE
results = []
for i, a in enumerate(sampled_alphas):
    t = DecisionTreeRegressor(ccp_alpha=a, random_state=42)
    t.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, t.predict(X_train))
    test_mse  = mean_squared_error(y_test,  t.predict(X_test))
    results.append({
        "alpha": a,
        "leaves": t.get_n_leaves(),
        "train_mse": train_mse,
        "test_mse": test_mse,
    })
    print(f"  [{i+1}/{len(sampled_alphas)}] alpha={a:.5f}, leaves={t.get_n_leaves()}, test MSE={test_mse:.4f}")

# Find the winner
best = min(results, key=lambda r: r["test_mse"])
print(f"\n*** BEST: alpha={best['alpha']:.5f}, leaves={best['leaves']}, test MSE={best['test_mse']:.4f} ***")
print(f"    (medium tree from step 3 had test MSE = 13.11)")

  [1/33] alpha=0.00000, leaves=235, test MSE=17.3038
  [2/33] alpha=0.00002, leaves=229, test MSE=17.2945
  [3/33] alpha=0.00008, leaves=223, test MSE=17.3470
  [4/33] alpha=0.00019, leaves=217, test MSE=17.3041
  [5/33] alpha=0.00019, leaves=211, test MSE=17.2869
  [6/33] alpha=0.00033, leaves=205, test MSE=17.2469
  [7/33] alpha=0.00035, leaves=198, test MSE=17.2421
  [8/33] alpha=0.00052, leaves=192, test MSE=17.2497
  [9/33] alpha=0.00075, leaves=186, test MSE=17.3039
  [10/33] alpha=0.00084, leaves=180, test MSE=17.3313
  [11/33] alpha=0.00120, leaves=174, test MSE=17.3953
  [12/33] alpha=0.00139, leaves=168, test MSE=17.4216
  [13/33] alpha=0.00176, leaves=162, test MSE=17.6352
  [14/33] alpha=0.00278, leaves=156, test MSE=17.6811
  [15/33] alpha=0.00367, leaves=150, test MSE=17.5369
  [16/33] alpha=0.00533, leaves=142, test MSE=17.6250
  [17/33] alpha=0.00625, leaves=136, test MSE=17.5684
  [18/33] alpha=0.00775, leaves=130, test MSE=17.6005
  [19/33] alpha=0.00889, leaves=122, 

In [27]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import pandas as pd

# Retrain the winning tree so we can inspect it
best_alpha = 0.27643  # uses the alpha that won
best_tree = DecisionTreeRegressor(ccp_alpha=best_alpha, random_state=42)
best_tree.fit(X_train, y_train)

importances = pd.Series(
    best_tree.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Feature importances (higher = more useful for prediction):\n")
importances

Feature importances (higher = more useful for prediction):



PTS    0.485484
FGA    0.201793
AST    0.184614
TOV    0.128110
dtype: float64

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

# dumb model: always predict the mean of training y
mean_pred = np.full(shape=len(y_test), fill_value=y_train.mean().item())
null_mse = mean_squared_error(y_test, mean_pred)

print(f"Null model (predict mean of y_train): test MSE = {null_mse:.4f}")
print(f"Your best pruned tree:                 test MSE = 10.49")
print(f"Improvement over null model:           {(null_mse - 10.49) / null_mse * 100:.1f}%")

Null model (predict mean of y_train): test MSE = 11.6591
Your best pruned tree:                 test MSE = 10.49
Improvement over null model:           10.0%


#### Levels of importance
1) Points
2) Field Goals Attempted
3) Assists
4) Turnovers

This makes total sense, points contribute to plus minus directly so this is not surprising.
FGA is also correlated to that and maybe could have been left out of training

In [34]:
Individual_Players = [
    'Rudy Gobert',
    'Victor Wembanyama',
    'Bam Adebayo',
    'Anthony Davis',
    'Herbert Jones',
    ]
sample = df[df['PLAYER_NAME'].isin(Individual_Players)][
    ['PLAYER_NAME', 'PTS', 'TOV', 'FGA', 'AST', 'PLUS_MINUS']
].copy()

X_sample = sample[['PTS', 'TOV', 'FGA', 'AST']]
sample['PREDICTED'] = best_tree.predict(X_sample)
sample['ERROR'] = sample['PLUS_MINUS'] - sample['PREDICTED']

print(sample.to_string(index=False))

      PLAYER_NAME  PTS  TOV  FGA  AST  PLUS_MINUS  PREDICTED     ERROR
    Anthony Davis 24.7  2.1 16.9  3.5         2.0   5.186667 -3.186667
      Bam Adebayo 19.3  2.3 14.3  3.9         0.7  -1.613333  2.313333
    Herbert Jones 11.0  1.2  7.7  2.6         3.0  -0.181507  3.181507
      Rudy Gobert 14.0  1.6  8.1  1.3         5.4   2.339130  3.060870
Victor Wembanyama 21.4  3.7 16.7  3.9        -2.0  -1.613333 -0.386667


These 5 players finished 1-5 in the 2023-24 defensive player of the year award
1) Rudy Gobert
2) Wembanyama
3) Adebayo
4) Davis
5) Jones

The model predicted all of them other than Davis to have a lower plus-minus than they really had.  Due to their defensive contributions they improve the plus-minus.  It would be interesting to take out scoring metrics and use only rebounds, steals, turnovers.  

# Summary and Takeaways

Predicted NBA player plus-minus from points, assists, turnovers, and field goals attempted using regression trees.  Conducted on a total of 301 players who played over 30 games with over 15 minutes per game in the 2023-24 NBA regular season.

Using cost-complexity post-pruning achieved test Mean Squared Error = 10.49, a 10% improvement on a dumb model that would just predict the average plus-minus for each player.

##### Limitations

Plus minus is just as much an offensive stat as it is a defensive stat.  So for a players with high defensive caliber the model will predict under their real plus-minus.  Plus minus is also skewed for extremely utilized players. Players like Nikola Jokic or Steph Curry have entire offenses run through them so when they are off the floor the minus part of plus minus will over-contribute.

Learning ML has been fun so far, using it to analyze NBA data is something that I am going to do more of it 
-WH